In [1]:
import importlib.metadata as metadata

try:
    print("LangGraph:", metadata.version("langgraph"))
except metadata.PackageNotFoundError:
    print("LangGraph 未安装")

LangGraph: 1.1.2


In [2]:
# 调用 LLM 生成答案
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.chat_models import ChatTongyi
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv
from rich import print as rprint
import time


load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

llmodel_deepseek = init_chat_model(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,

)



In [3]:
from langchain_core.tools import tool

@tool
def calculate_effective_rainfall(
    precipitation_mm: float,
    infiltration_mm: float
) -> float:
    """
    计算有效降雨量。

    Args:
        precipitation_mm: 总降水量，单位mm。
        infiltration_mm: 入渗量，单位mm。

    Returns:
        有效降雨量，单位mm。
    """

    if precipitation_mm < 0 or infiltration_mm < 0:
        raise ValueError("降水量和入渗量不能为负数")

    if infiltration_mm > precipitation_mm:
        raise ValueError("入渗量不能大于总降水量")

    return precipitation_mm - infiltration_mm

tools = [
    calculate_effective_rainfall
]

for t in tools:
    print(t.name)
    print(t.description)

@tool
def calculate_rainfall_statistics(
    precipitation: list[float]
) -> dict:
    """
    统计一段时间内的降雨数据。

    Args:
        precipitation: 每日降水量列表，单位mm。

    Returns:
        总降水量、平均降水量、最大日降水量、
        降雨天数。
    """

    if not precipitation:
        raise ValueError("降水数据不能为空")

    if any(p < 0 for p in precipitation):
        raise ValueError("降水量不能为负数")

    return {
        "total_rainfall": sum(precipitation),
        "mean_rainfall": sum(precipitation) / len(precipitation),
        "max_rainfall": max(precipitation),
        "rainy_days": sum(p > 0.1 for p in precipitation),
        "total_days": len(precipitation)
    }


from langchain_core.messages import ToolMessage

# 建立工具名称与工具对象的映射
tool_map = {
    tool.name: tool
    for tool in tools
}

# # 获取模型生成的工具调用
# for tool_call in response.tool_calls:

#     tool_name = tool_call["name"]
#     tool_args = tool_call["args"]

#     # 找到对应工具
#     selected_tool = tool_map[tool_name]

#     # 执行工具
#     tool_result = selected_tool.invoke(tool_args)

#     print("工具名称:", tool_name)
#     print("工具参数:", tool_args)
#     print("执行结果:", tool_result)

calculate_effective_rainfall
计算有效降雨量。

Args:
    precipitation_mm: 总降水量，单位mm。
    infiltration_mm: 入渗量，单位mm。

Returns:
    有效降雨量，单位mm。


In [4]:
tools = [
    calculate_effective_rainfall,
    calculate_rainfall_statistics
]

In [5]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import (
    AnyMessage,
    SystemMessage,
    HumanMessage
)

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.graph.message import add_messages

from langgraph.prebuilt import ToolNode, tools_condition

LangGraph 的四个核心概念
|概念|含义|在项目中的作用
|---|---|---|
|State|状态|保存消息、具结果等
|Node|节点|执行模型或工具
|Edge|边|控制执行顺序
|Conditional Edge|条件边|根据模型结果决定下一步

这是 LangGraph 非常重要的一个概念。

State 就是工作流的状态容器。

其中：
- messages：保存用户消息。
- 模型回复：保存在消息列表中。
- ToolMessage：保存工具执行结果。
add_messages 表示新消息会追加到已有消息历史，而不是直接覆盖。

In [6]:

# 定义状态State
class AgentState(TypedDict):
    messages: Annotated[
        list[AnyMessage],
        add_messages
    ]

In [7]:
#绑定工具
llm_with_tools = llmodel_deepseek.bind_tools(
    tools
)

这个节点负责：
1. 获取当前状态。
2. 将消息发送给 LLM。
3. 获取模型回复。
4. 将回复写回 State

In [8]:
def agent_node(state: AgentState):

    response = llm_with_tools.invoke(
        state["messages"],
        temperature=0
    )

    return {
        "messages": [response]
    }

In [10]:
####构建工作流
tool_node = ToolNode(tools)

builder = StateGraph(AgentState)

# 注册节点
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

# 设置入口
builder.add_edge(START, "agent")

# 根据模型是否调用工具进行条件路由
builder.add_conditional_edges(
    "agent",
    tools_condition
)

# 工具执行后返回 Agent
builder.add_edge("tools", "agent")

# 编译
graph = builder.compile()

print("LangGraph Agent 构建成功！")

LangGraph Agent 构建成功！


In [13]:
from rich import print as rprint
#测试
query = """
某地区连续7天的降雨量为：

[12, 25, 0, 38, 45, 8, 15] mm。

整个降雨过程的总入渗量为35mm。

请完成：
1. 统计总降雨量、平均日降雨量和最大日降雨量。
2. 计算有效降雨量。
3. 解释本次降雨过程的特点。
"""

initial_state = {
    "messages": [
        SystemMessage(
            content=(
                "你是专业的水文气象分析助手。"
                "需要计算时必须调用对应工具。"
                "只能依据工具实际返回的数据进行分析。"
                "不能将有效降雨量直接等同于实际径流量。"
            )
        ),
        HumanMessage(content=query)
    ]
}

result = graph.invoke(
    initial_state,
    config={
        "recursion_limit": 10
    }
)

rprint(result["messages"][-1].content)

## 一、降雨统计结果

| 统计项 | 数值 |
|---|---|
| 总降雨量 | **143.0 mm** |
| 平均日降雨量 | **20.43 mm/天**（按7天计） |
| 最大日降雨量 | **45.0 mm**（第5天） |
| 降雨天数 | **6 天**（7天中有1天无雨） |

> 说明：平均日降雨量按7天平均计算，为 143 ÷ 7 ≈ 20.43 mm。扣除无雨日后，若按6个降雨日计算，则降雨日平均约为 23.83 
mm/天。

## 二、有效降雨量

- 总降水量：143 mm
- 总入渗量：35 mm
- **有效降雨量 = 143 − 35 = 108.0 mm**

## 三、本次降雨过程特点

1. **总量较充沛、时段集中**：7天累计143 mm，属于一次中等偏强的降雨过程；雨量主要集中在中后期。
2. **分布不均，呈“双峰”形态**：第4天38 mm、第5天45 mm连续两天大雨，两日合计83 
mm，占总量的58%，是全过程的降雨核心期；前后期仅为12、25、8、15 mm的小到中雨。
3. **存在明显的降雨间歇**：第3天为0 mm，第6—7天雨量回落至8 mm和15 
mm，说明过程并非连续强降雨，而是“前期铺垫—中期主峰—后期收尾”的结构。
4. **有效降雨占比高**：有效降雨量108 mm，占总量约75.5%，入渗35 
mm约占24.5%。这表明在下垫面入渗能力有限的情况下，本次降雨有相当大比例可能转化为地表径流，存在一定的产流和积涝风险，
尤其在雨量最集中的第4—5天。

⚠️ 
需要说明的是：有效降雨量（降水扣除入渗）只是产流潜力的一个近似指标，**并不等同于实际径流量**。实际径流量还会受到土
壤前期含水量、地形坡度、植被覆盖、蒸发、洼地蓄水及汇流过程等因素影响，需结合流域产流模型或实测资料进一步分析。

In [14]:
# 查看整个 Agent 执行过程
for message in result["messages"]:

    print("\n", "=" * 60)

    print("Message Type:", type(message).__name__)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool Calls:", message.tool_calls)

    elif hasattr(message, "tool_call_id"):
        print("Tool Result:", message.content)

    else:
        print("Content:", message.content)


Message Type: SystemMessage
Content: 你是专业的水文气象分析助手。需要计算时必须调用对应工具。只能依据工具实际返回的数据进行分析。不能将有效降雨量直接等同于实际径流量。

Message Type: HumanMessage
Content: 
某地区连续7天的降雨量为：

[12, 25, 0, 38, 45, 8, 15] mm。

整个降雨过程的总入渗量为35mm。

请完成：
1. 统计总降雨量、平均日降雨量和最大日降雨量。
2. 计算有效降雨量。
3. 解释本次降雨过程的特点。


Message Type: AIMessage
Tool Calls: [{'name': 'calculate_rainfall_statistics', 'args': {'precipitation': [12, 25, 0, 38, 45, 8, 15]}, 'id': 'call_00_kLVSV9Wsw7tXY3zQhFVv9433', 'type': 'tool_call'}, {'name': 'calculate_effective_rainfall', 'args': {'precipitation_mm': 143, 'infiltration_mm': 35}, 'id': 'call_01_IO2bPvzE90oP8yqnlaBk8715', 'type': 'tool_call'}]

Message Type: ToolMessage
Tool Result: {"total_rainfall": 143.0, "mean_rainfall": 20.428571428571427, "max_rainfall": 45.0, "rainy_days": 6, "total_days": 7}

Message Type: ToolMessage
Tool Result: 108.0

Message Type: AIMessage
Content: ## 一、降雨统计结果

| 统计项 | 数值 |
|---|---|
| 总降雨量 | **143.0 mm** |
| 平均日降雨量 | **20.43 mm/天**（按7天计） |
| 最大日降雨量 | **45.0 mm**（第5天） |
| 降雨天数 |

In [19]:
from langgraph.checkpoint.memory import MemorySaver

# 创建内存检查点
memory = MemorySaver()

# 重新编译工作流
graph_memory = builder.compile(checkpointer= memory)

print("Checkpoint 初始化成功.")

Checkpoint 初始化成功.


In [17]:
# 创建会话ID
config = {
    "configurable": {
        "thread_id": "rainfall_session_001"
    }
}

In [20]:
# 第一次对话
response1 = graph_memory.invoke(
    {
        "messages": [
            SystemMessage(
                content=(
                    "你是专业的水文气象分析助手。"
                    "需要计算时必须调用工具。"
                    "不能编造计算结果。"
                )
            ),
            HumanMessage(
                content="某地区7天降雨量为[12,25,0,38,45,8,15]，总降雨量是多少？"
            )
        ]
    },
    config=config
)

print(response1["messages"][-1].content)

## 计算结果显示，这7天总降雨量为 **143 mm**。

完整的统计信息如下：

| 指标 | 数值 |
|------|------|
| 总降水量 | **143.0 mm** |
| 平均降水量，按7天计算 | 20.43 mm/天 |
| 最大日降水量 | 45.0 mm（第5天） |
| 降雨天数 | 6 天（仅第3天无雨） |

**逐日明细：** 12 + 25 + 0 + 38 + 45 + 8 + 15 = 143 mm

**水情提示：** 该时段降雨分布不均，第4~5天连续降雨量合计 83 mm（占总量 58%），属于主要产流时段，需关注短时强降雨可能引发的径流集中和下垫面饱和问题。若需要进一步计算有效降雨量（扣除入渗部分）或面雨量，请提供入渗参数或流域面积。


In [21]:
#第二次对话
response2 = graph_memory.invoke(
    {
        "messages": [
            HumanMessage(
                content="如果这次降雨过程的总入渗量为35mm，那么有效降雨量是多少？"
            )
        ]
    },
    config=config
)

print(response2["messages"][-1].content)

## 有效降雨量 = **108.0 mm**

**计算过程：**

- 总降水量：143 mm
- 总入渗量：35 mm
- 有效降雨量 = 143 − 35 = **108 mm**

**产流分析：**

| 项目 | 数值 |
|------|------|
| 总降水量 | 143 mm |
| 入渗量 | 35 mm |
| 有效降雨量 | 108 mm |
| 产流系数（径流系数） | 108 / 143 ≈ **0.755** |

径流系数约为 0.76，属于偏高的水平。这表示约 76% 的降雨转化为地表径流或产流量，只有 24% 通过入渗补给土壤和地下水。结合上一次分析中第4~5天的集中强降雨（83 mm、占比58%），土壤前期可能已接近饱和，因此实际场次入渗量可能比采用固定值35 mm估算的结果更小，而产流量可能偏高。

**说明与建议：**

1. 该结果基于**总量扣减法**，适合对场次或长历时降雨进行初步估算。
2. 如果降雨强度变化大，建议采用**下渗曲线法（如 Horton、Green-Ampt 或 SCS-CN 模型）逐时段计算**，这样的结果会更准确；尤其该场次存在明显强降雨集中段，总量法的误差可能较大。
3. 如需评估洪水风险，还需结合流域面积将其换算为径流总量或净雨深；我可以继续协助换算。


In [22]:
#查看保存的状态
state = graph_memory.get_state(config)

print("当前会话消息数量：", len(state.values["messages"]))

for message in state.values["messages"]:
    print(
        type(message).__name__,
        str(message.content)[:200]
    )

当前会话消息数量： 9
SystemMessage 你是专业的水文气象分析助手。需要计算时必须调用工具。不能编造计算结果。
HumanMessage 某地区7天降雨量为[12,25,0,38,45,8,15]，总降雨量是多少？
AIMessage 我来计算这7天的降雨统计。
ToolMessage {"total_rainfall": 143.0, "mean_rainfall": 20.428571428571427, "max_rainfall": 45.0, "rainy_days": 6, "total_days": 7}
AIMessage ## 计算结果显示，这7天总降雨量为 **143 mm**。

完整的统计信息如下：

| 指标 | 数值 |
|------|------|
| 总降水量 | **143.0 mm** |
| 平均降水量，按7天计算 | 20.43 mm/天 |
| 最大日降水量 | 45.0 mm（第5天） |
| 降雨天数 | 6 天（仅第3天无雨） |

**逐日明细：** 12 + 25 + 0 + 3
HumanMessage 如果这次降雨过程的总入渗量为35mm，那么有效降雨量是多少？
AIMessage 
ToolMessage 108.0
AIMessage ## 有效降雨量 = **108.0 mm**

**计算过程：**

- 总降水量：143 mm
- 总入渗量：35 mm
- 有效降雨量 = 143 − 35 = **108 mm**

**产流分析：**

| 项目 | 数值 |
|------|------|
| 总降水量 | 143 mm |
| 入渗量 | 35 mm |
| 有效降雨量 | 108 mm |
| 产流系数（径流系数）


In [23]:

# 创建新的测试工作流

from typing import Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END


class TaskState(TypedDict):
    query: str
    task_type: str
    result: str

In [24]:

# 定义分类节点
def classify_node(state: TaskState):

    query = state["query"]

    prompt = f"""
    判断用户问题属于以下哪种类型：

    1. retrieval：知识检索
    2. calculation：数值计算
    3. analysis：数据分析

    用户问题：
    {query}

    只返回一个英文类别名称。
    """

    response = llmodel_deepseek.invoke(prompt)

    task_type = response.content.strip().lower()

    if task_type not in [
        "retrieval",
        "calculation",
        "analysis"
    ]:
        task_type = "analysis"

    return {
        "task_type": task_type
    }

In [25]:
def retrieval_node(state: TaskState):

    return {
        "result": "执行知识库检索任务：" + state["query"]
    }


def calculation_node(state: TaskState):

    return {
        "result": "执行数值计算任务：" + state["query"]
    }


def analysis_node(state: TaskState):

    return {
        "result": "执行数据分析任务：" + state["query"]
    }

In [26]:
#定义条件路由
def route_task(
    state: TaskState
) -> Literal["retrieval", "calculation", "analysis"]:

    task_type = state["task_type"]

    if task_type == "retrieval":
        return "retrieval"

    elif task_type == "calculation":
        return "calculation"

    else:
        return "analysis"

In [28]:
builder = StateGraph(TaskState)

builder.add_node("classify", classify_node)
builder.add_node("retrieval", retrieval_node)
builder.add_node("calculation", calculation_node)
builder.add_node("analysis", analysis_node)

builder.add_edge(START, "classify")

builder.add_conditional_edges(
    "classify",
    route_task
)

builder.add_edge("retrieval", END)
builder.add_edge("calculation", END)
builder.add_edge("analysis", END)

conditional_graph = builder.compile()

print("条件路由工作流构建成功！")



test_queries = [
    "地下水位升高为什么会影响滑坡稳定性？",
    "总降雨量为120mm，入渗量为35mm，有效降雨量是多少？",
    "分析过去7天的降雨变化特征。"
]

for query in test_queries:

    result = conditional_graph.invoke({
        "query": query,
        "task_type": "",
        "result": ""
    })

    print("\n问题：", query)
    print("任务类型：", result["task_type"])
    print("执行结果：", result["result"])

条件路由工作流构建成功！

问题： 地下水位升高为什么会影响滑坡稳定性？
任务类型： retrieval
执行结果： 执行知识库检索任务：地下水位升高为什么会影响滑坡稳定性？

问题： 总降雨量为120mm，入渗量为35mm，有效降雨量是多少？
任务类型： calculation
执行结果： 执行数值计算任务：总降雨量为120mm，入渗量为35mm，有效降雨量是多少？

问题： 分析过去7天的降雨变化特征。
任务类型： analysis
执行结果： 执行数据分析任务：分析过去7天的降雨变化特征。


In [29]:
from typing import Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
class ApprovalState(TypedDict):
    operation: str
    approval: str
    result: str

def approval_node(state: ApprovalState):

    decision = interrupt({
        "message": "请确认是否执行以下操作",
        "operation": state["operation"]
    })

    return {
        "approval": decision
    }

def execute_node(state: ApprovalState):

    operation = state["operation"]

    return {
        "result": f"已执行批准的操作：{operation}"
    }


def reject_node(state: ApprovalState):

    return {
        "result": f"操作已被拒绝，未执行：{state['operation']}"
    }

def approval_router(
    state: ApprovalState
) -> Literal["execute", "reject"]:

    if state["approval"] == "approve":
        return "execute"

    return "reject"


approval_builder = StateGraph(ApprovalState)

approval_builder.add_node("approval", approval_node)
approval_builder.add_node("execute", execute_node)
approval_builder.add_node("reject", reject_node)

approval_builder.add_edge(START, "approval")

approval_builder.add_conditional_edges(
    "approval",
    approval_router
)

approval_builder.add_edge("execute", END)
approval_builder.add_edge("reject", END)

approval_graph = approval_builder.compile(
    checkpointer=MemorySaver()
)

print("人工审批工作流构建成功！")

人工审批工作流构建成功！


In [30]:
approval_config = {
    "configurable": {
        "thread_id": "approval_test_001"
    }
}

approval_result = approval_graph.invoke(
    {
        "operation": "删除2024年之前的知识库数据",
        "approval": "",
        "result": ""
    },
    config=approval_config
)

print(approval_result)

{'operation': '删除2024年之前的知识库数据', 'approval': '', 'result': '', '__interrupt__': [Interrupt(value={'message': '请确认是否执行以下操作', 'operation': '删除2024年之前的知识库数据'}, id='d293b820fa6e1f1e00fd205d8a6f2c3a')]}


In [32]:
state = approval_graph.get_state(
    approval_config
)

print("当前状态：", state.values)
print("是否暂停：", state.next)

approval_result = approval_graph.invoke(
    Command(resume="approve"),
    config=approval_config
)

print(approval_result["result"])

当前状态： {'operation': '删除2024年之前的知识库数据', 'approval': '', 'result': ''}
是否暂停： ('approval',)
已执行批准的操作：删除2024年之前的知识库数据


In [33]:
reject_config = {
    "configurable": {
        "thread_id": "approval_test_002"
    }
}

approval_graph.invoke(
    {
        "operation": "删除2024年之前的知识库数据",
        "approval": "",
        "result": ""
    },
    config=reject_config
)

reject_result = approval_graph.invoke(
    Command(resume="reject"),
    config=reject_config
)

print(reject_result["result"])

操作已被拒绝，未执行：删除2024年之前的知识库数据
